In [11]:
%%configure -f
{
  "defaultLakehouse": {
    "name": "lh_insurance_bronze_silver",
    "id": "c5d42bb8-2bbc-4172-a40b-8fbcdee30e29",
    "workspaceId": "3533e4eb-ba19-4350-b072-e6497106fb81"
  }
}

StatementMeta(, a8cfbf21-b901-43d6-8524-03f284f5b077, -1, Finished, Available, Finished, True)

In [12]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

StatementMeta(, a8cfbf21-b901-43d6-8524-03f284f5b077, 3, Finished, Available, Finished, False)

In [13]:
claims_bronze = spark.table("bronze_claims_raw")
policies_bronze = spark.table("bronze_policies_raw")
customers_bronze = spark.table("bronze_customers_raw")
vehicles_bronze = spark.table("bronze_vehicles_raw")
payments_bronze = spark.table("bronze_payments_raw")

StatementMeta(, a8cfbf21-b901-43d6-8524-03f284f5b077, 4, Finished, Available, Finished, False)

In [14]:
claims_silver = (
    claims_bronze
    .withColumn("claim_number", F.upper(F.trim(F.col("claim_number"))))
    .withColumn("policy_number", F.upper(F.trim(F.col("policy_number"))))
    .withColumn("vin", F.upper(F.trim(F.col("vin"))))
    .withColumn("claimant_customer_id", F.upper(F.trim(F.col("claimant_customer_id"))))
    .withColumn("incident_date", F.to_date("incident_date"))
    .withColumn("reported_date", F.to_date("reported_date"))
    .withColumn("claim_status", F.initcap(F.trim(F.col("claim_status"))))
    .withColumn("loss_type", F.initcap(F.trim(F.col("loss_type"))))
    .withColumn("fraud_indicator", F.upper(F.trim(F.col("fraud_indicator"))))
    .withColumn("claim_amount", F.col("claim_amount").cast("decimal(12,2)"))
    .withColumn("reserve_amount", F.col("reserve_amount").cast("decimal(12,2)"))
    .withColumn("days_to_report", F.datediff(F.col("reported_date"), F.col("incident_date")))
    .dropDuplicates(["claim_number"])
)

StatementMeta(, a8cfbf21-b901-43d6-8524-03f284f5b077, 5, Finished, Available, Finished, False)

In [15]:
policies_silver = (
    policies_bronze
    .withColumn("policy_number", F.upper(F.trim(F.col("policy_number"))))
    .withColumn("customer_id", F.upper(F.trim(F.col("customer_id"))))
    .withColumn("policy_start_date", F.to_date("policy_start_date"))
    .withColumn("policy_end_date", F.to_date("policy_end_date"))
    .withColumn("coverage_type", F.initcap(F.trim(F.col("coverage_type"))))
    .withColumn("underwriting_status", F.initcap(F.trim(F.col("underwriting_status"))))
    .withColumn("state_code", F.upper(F.trim(F.col("state_code"))))
    .withColumn("deductible_amount", F.col("deductible_amount").cast("decimal(12,2)"))
    .dropDuplicates(["policy_number"])
)

StatementMeta(, a8cfbf21-b901-43d6-8524-03f284f5b077, 6, Finished, Available, Finished, False)

In [16]:
customers_silver = (
    customers_bronze
    .withColumn("customer_id", F.upper(F.trim(F.col("customer_id"))))
    .withColumn("full_name", F.initcap(F.trim(F.col("full_name"))))
    .withColumn("date_of_birth", F.to_date("date_of_birth"))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn("city", F.initcap(F.trim(F.col("city"))))
    .withColumn("state_code", F.upper(F.trim(F.col("state_code"))))
    .withColumn("postal_code", F.trim(F.col("postal_code")))
    .withColumn("risk_segment", F.initcap(F.trim(F.col("risk_segment"))))
    .dropDuplicates(["customer_id"])
)

StatementMeta(, a8cfbf21-b901-43d6-8524-03f284f5b077, 7, Finished, Available, Finished, False)

In [17]:
vehicles_silver = (
    vehicles_bronze
    .withColumn("vin", F.upper(F.trim(F.col("vin"))))
    .withColumn("policy_number", F.upper(F.trim(F.col("policy_number"))))
    .withColumn("vehicle_make", F.initcap(F.trim(F.col("vehicle_make"))))
    .withColumn("vehicle_model", F.initcap(F.trim(F.col("vehicle_model"))))
    .withColumn("vehicle_year", F.col("vehicle_year").cast("int"))
    .withColumn("usage_type", F.initcap(F.trim(F.col("usage_type"))))
    .withColumn("garaging_state", F.upper(F.trim(F.col("garaging_state"))))
    .dropDuplicates(["vin"])
)

StatementMeta(, a8cfbf21-b901-43d6-8524-03f284f5b077, 8, Finished, Available, Finished, False)

In [18]:
payments_silver = (
    payments_bronze
    .withColumn("payment_number", F.upper(F.trim(F.col("payment_number"))))
    .withColumn("claim_number", F.upper(F.trim(F.col("claim_number"))))
    .withColumn("payment_date", F.to_date("payment_date"))
    .withColumn("payment_amount", F.col("payment_amount").cast("decimal(12,2)"))
    .withColumn("payment_method", F.initcap(F.trim(F.col("payment_method"))))
    .withColumn("payment_status", F.initcap(F.trim(F.col("payment_status"))))
    .dropDuplicates(["payment_number"])
)

StatementMeta(, a8cfbf21-b901-43d6-8524-03f284f5b077, 9, Finished, Available, Finished, False)

In [19]:
claims_silver = claims_silver.withColumn(
    "claim_hashdiff",
    F.sha2(F.concat_ws("|",
        F.col("claim_status"),
        F.col("loss_type"),
        F.col("claim_amount").cast("string"),
        F.col("reserve_amount").cast("string"),
        F.col("fraud_indicator")
    ), 256)
)

policies_silver = policies_silver.withColumn(
    "policy_hashdiff",
    F.sha2(F.concat_ws("|",
        F.col("coverage_type"),
        F.col("deductible_amount").cast("string"),
        F.col("underwriting_status"),
        F.col("state_code")
    ), 256)
)

customers_silver = customers_silver.withColumn(
    "customer_hashdiff",
    F.sha2(F.concat_ws("|",
        F.col("full_name"),
        F.col("email"),
        F.col("city"),
        F.col("state_code"),
        F.col("risk_segment")
    ), 256)
)

vehicles_silver = vehicles_silver.withColumn(
    "vehicle_hashdiff",
    F.sha2(F.concat_ws("|",
        F.col("vehicle_make"),
        F.col("vehicle_model"),
        F.col("vehicle_year").cast("string"),
        F.col("usage_type"),
        F.col("garaging_state")
    ), 256)
)

payments_silver = payments_silver.withColumn(
    "payment_hashdiff",
    F.sha2(F.concat_ws("|",
        F.col("payment_date").cast("string"),
        F.col("payment_amount").cast("string"),
        F.col("payment_method"),
        F.col("payment_status")
    ), 256)
)

StatementMeta(, a8cfbf21-b901-43d6-8524-03f284f5b077, 10, Finished, Available, Finished, False)

In [20]:
claims_silver.write.mode("overwrite").format("delta").saveAsTable("silver_stg_claims")
policies_silver.write.mode("overwrite").format("delta").saveAsTable("silver_stg_policies")
customers_silver.write.mode("overwrite").format("delta").saveAsTable("silver_stg_customers")
vehicles_silver.write.mode("overwrite").format("delta").saveAsTable("silver_stg_vehicles")
payments_silver.write.mode("overwrite").format("delta").saveAsTable("silver_stg_payments")

StatementMeta(, a8cfbf21-b901-43d6-8524-03f284f5b077, 11, Finished, Available, Finished, False)

In [21]:
for table_name in [
    "silver_stg_claims",
    "silver_stg_policies",
    "silver_stg_customers",
    "silver_stg_vehicles",
    "silver_stg_payments"
]:
    print(f"\n=== {table_name} ===")
    spark.sql(f"SELECT COUNT(*) AS row_count FROM {table_name}").show()

StatementMeta(, a8cfbf21-b901-43d6-8524-03f284f5b077, 12, Finished, Available, Finished, False)


=== silver_stg_claims ===
+---------+
|row_count|
+---------+
|       12|
+---------+


=== silver_stg_policies ===
+---------+
|row_count|
+---------+
|        8|
+---------+


=== silver_stg_customers ===
+---------+
|row_count|
+---------+
|        8|
+---------+


=== silver_stg_vehicles ===
+---------+
|row_count|
+---------+
|        8|
+---------+


=== silver_stg_payments ===
+---------+
|row_count|
+---------+
|       10|
+---------+

